# Retail Demand Forecasting - Step 1: Data Collection

This notebook implements **Step 1 (Data Collection)** of the standard ML lifecycle.

Collection strategy:
- Primary source: **Neon PostgreSQL** (live transactional extraction)
- Fallback source: local raw CSV in `data/raw/`
- Output: standardized raw dataset for downstream preprocessing and feature engineering

## Required Environment Variables
- `NEON_DATABASE_URL` (preferred) or `DATABASE_URL`
- Optional: `NEON_SOURCE_QUERY` to override the default SQL extraction query

You can store these in a `.env` file at the ML engine root.

In [1]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 150)

In [2]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FALLBACK_PATH = RAW_DIR / "DISSANAYAKA POS DATASET.csv"
NEON_SNAPSHOT_PATH = RAW_DIR / "pos_transactions_neon_snapshot.csv"

ENV_PATH = PROJECT_ROOT / ".env"
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)

DATABASE_URL = os.getenv("NEON_DATABASE_URL") or os.getenv("DATABASE_URL")

DEFAULT_QUERY = """
SELECT
    s.receipt_no AS "TransactionID",
    TO_CHAR(s.sale_date, 'DD/MM/YYYY') AS "Date",
    TO_CHAR(s.sale_date, 'HH24:MI:SS') AS "Time",
    COALESCE(si.product_id::text, p.sku, si.product_name) AS "ProductID",
    COALESCE(p.product_name, si.product_name) AS "ProductName",
    COALESCE(p.category, 'Unknown') AS "Category",
    COALESCE(p.unit, 'Unit') AS "PricingUnit",
    COALESCE(si.unit_price, p.selling_price::double precision, 0) AS "UnitPrice",
    COALESCE(p.buying_price::double precision, 0) AS "BuyingPrice",
    COALESCE(p.selling_price::double precision, si.unit_price, 0) AS "SellingPrice",
    COALESCE(si.quantity::double precision, 0) AS "Quantity",
    COALESCE(s.total_amount, 0) AS "Total_LKR",
    COALESCE(s.payment_method, 'Unknown') AS "Payment Method"
FROM sales s
JOIN sale_items si ON si.sale_id = s.id
LEFT JOIN products p ON p.id = si.product_id
ORDER BY s.sale_date ASC;
"""

QUERY = os.getenv("NEON_SOURCE_QUERY") or DEFAULT_QUERY

print("Project root:", PROJECT_ROOT)
print("Local fallback file exists:", LOCAL_FALLBACK_PATH.exists())
print("Neon URL configured:", bool(DATABASE_URL))

Project root: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine
Local fallback file exists: True
Neon URL configured: False


In [3]:
def fetch_from_neon(database_url: str, query: str) -> pd.DataFrame:
    if not database_url:
        raise ValueError("DATABASE_URL is missing.")

    engine = create_engine(database_url, pool_pre_ping=True)
    try:
        with engine.connect() as conn:
            df = pd.read_sql(text(query), conn)
    finally:
        engine.dispose()

    return df


def load_step1_raw_dataset() -> tuple[pd.DataFrame, str]:
    if DATABASE_URL:
        try:
            neon_df = fetch_from_neon(DATABASE_URL, QUERY)
            neon_df.to_csv(NEON_SNAPSHOT_PATH, index=False)
            return neon_df, "neon_db"
        except (SQLAlchemyError, ValueError, Exception) as ex:
            print(f"Neon fetch failed. Falling back to local CSV. Reason: {ex}")

    if LOCAL_FALLBACK_PATH.exists():
        fallback_df = pd.read_csv(LOCAL_FALLBACK_PATH, low_memory=False)
        return fallback_df, "local_csv_fallback"

    raise FileNotFoundError(
        f"No data source available. Configure Neon DB or provide fallback file: {LOCAL_FALLBACK_PATH}"
    )

In [4]:
raw_df, data_source = load_step1_raw_dataset()

required_cols = [
    "Date",
    "Time",
    "ProductID",
    "ProductName",
    "Quantity",
    "Total_LKR",
]

missing_cols = [c for c in required_cols if c not in raw_df.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")

raw_df["Quantity"] = pd.to_numeric(raw_df["Quantity"], errors="coerce")
raw_df["Total_LKR"] = pd.to_numeric(raw_df["Total_LKR"], errors="coerce")
raw_df = raw_df.dropna(subset=["Date", "Time", "ProductID", "ProductName", "Quantity"]).copy()

print(f"Data source: {data_source}")
print(f"Rows: {len(raw_df):,}")
print(f"Unique products: {raw_df['ProductID'].nunique():,}")
raw_df.head()

Data source: local_csv_fallback
Rows: 812,258
Unique products: 3,833


,TransactionID,Date,Time,ProductID,ProductName,Category,PricingUnit,UnitPrice,BuyingPrice,SellingPrice,Quantity,Total_LKR,Payment Method
0,TX0005400000,1/1/2021,7:47:47,PI00469,Whiskas Adult Ocean Fish Food - 500g,Pet Products,Pack,820,700,820,1,970,Cash
1,TX0005400000,1/1/2021,7:47:47,PI00097,Kotmale Milk Chocolate - 170ml,Dairy,Pack,50,45,150,3,970,Cash
2,TX0005400001,1/1/2021,8:08:33,PI03359,Precare Unisex Sandal 1 S07 Bj0003 - 1pc,Health & Beauty,Unit,5500,4600,5500,1,5980,Cash
3,TX0005400001,1/1/2021,8:08:33,PI03268,Newtro Chili Powder - 100g,Seeds & Spices,Pack,190,170,380,2,5980,Cash
4,TX0005400001,1/1/2021,8:08:33,PI00173,Denta Comfort Soft Toothbrushes - 2pcs,Health & Beauty,Unit,100,80,100,1,5980,Cash


In [5]:
STEP1_OUT = RAW_DIR / "step1_collected_dataset.csv"
raw_df.to_csv(STEP1_OUT, index=False)
print(f"Step 1 dataset saved to: {STEP1_OUT.resolve()}")

Step 1 dataset saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\data\raw\step1_collected_dataset.csv


## Step 2 - Data Preprocessing

This section performs production-style preprocessing on the Step 1 collected dataset.

Preprocessing actions:
- Build a reliable timestamp from Date and Time
- Remove invalid and duplicate rows
- Handle missing numeric fields with robust fallback logic
- Clip extreme outliers in Quantity and Total_LKR using IQR bounds
- Save cleaned dataset for feature engineering in Step 3

In [6]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

STEP2_OUT = PROCESSED_DIR / "step2_preprocessed_dataset.csv"


def _iqr_clip_bounds(series: pd.Series, factor: float = 1.5) -> tuple[float, float]:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    if pd.isna(iqr) or iqr == 0:
        return float(series.min()), float(series.max())

    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return float(lower), float(upper)


def preprocess_step2(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    work = frame.copy()

    report = {
        "rows_in": int(len(work)),
        "rows_after_timestamp": 0,
        "rows_after_dedup": 0,
        "rows_after_quality": 0,
        "quantity_outliers_clipped": 0,
        "total_lkr_outliers_clipped": 0,
    }

    work["timestamp"] = pd.to_datetime(
        work["Date"].astype(str) + " " + work["Time"].astype(str),
        dayfirst=True,
        errors="coerce",
    )

    work["Quantity"] = pd.to_numeric(work["Quantity"], errors="coerce")
    work["Total_LKR"] = pd.to_numeric(work["Total_LKR"], errors="coerce")

    work = work.dropna(subset=["timestamp", "ProductID", "ProductName", "Quantity"]).copy()
    report["rows_after_timestamp"] = int(len(work))

    dedup_keys = ["timestamp", "ProductID", "ProductName", "Quantity", "Total_LKR"]
    work = work.drop_duplicates(subset=dedup_keys).copy()
    report["rows_after_dedup"] = int(len(work))

    work = work[work["Quantity"] > 0].copy()

    if "Total_LKR" in work.columns:
        work["Total_LKR"] = work["Total_LKR"].fillna(work["Quantity"] * work.get("UnitPrice", 0))
        work["Total_LKR"] = work["Total_LKR"].fillna(0)

    q_low, q_high = _iqr_clip_bounds(work["Quantity"])
    q_before = work["Quantity"].copy()
    work["Quantity"] = work["Quantity"].clip(lower=max(0, q_low), upper=q_high)
    report["quantity_outliers_clipped"] = int((q_before != work["Quantity"]).sum())

    if "Total_LKR" in work.columns and work["Total_LKR"].notna().any():
        t_low, t_high = _iqr_clip_bounds(work["Total_LKR"])
        t_before = work["Total_LKR"].copy()
        work["Total_LKR"] = work["Total_LKR"].clip(lower=max(0, t_low), upper=t_high)
        report["total_lkr_outliers_clipped"] = int((t_before != work["Total_LKR"]).sum())

    work = work.sort_values(["timestamp", "ProductID"]).reset_index(drop=True)
    report["rows_after_quality"] = int(len(work))

    return work, report

In [7]:
step2_df, step2_report = preprocess_step2(raw_df)

print("Step 2 Report")
for key, value in step2_report.items():
    print(f"- {key}: {value:,}" if isinstance(value, int) else f"- {key}: {value}")

step2_df.head()

Step 2 Report
- rows_in: 812,258
- rows_after_timestamp: 812,258
- rows_after_dedup: 812,258
- rows_after_quality: 812,258
- quantity_outliers_clipped: 0
- total_lkr_outliers_clipped: 38,718


,TransactionID,Date,Time,ProductID,ProductName,Category,PricingUnit,UnitPrice,BuyingPrice,SellingPrice,Quantity,Total_LKR,Payment Method,timestamp
0,TX0005400000,1/1/2021,7:47:47,PI00097,Kotmale Milk Chocolate - 170ml,Dairy,Pack,50,45,150,3,970,Cash,2021-01-01 07:47:47
1,TX0005400000,1/1/2021,7:47:47,PI00469,Whiskas Adult Ocean Fish Food - 500g,Pet Products,Pack,820,700,820,1,970,Cash,2021-01-01 07:47:47
2,TX0005400001,1/1/2021,8:08:33,PI00173,Denta Comfort Soft Toothbrushes - 2pcs,Health & Beauty,Unit,100,80,100,1,5980,Cash,2021-01-01 08:08:33
3,TX0005400001,1/1/2021,8:08:33,PI03268,Newtro Chili Powder - 100g,Seeds & Spices,Pack,190,170,380,2,5980,Cash,2021-01-01 08:08:33
4,TX0005400001,1/1/2021,8:08:33,PI03359,Precare Unisex Sandal 1 S07 Bj0003 - 1pc,Health & Beauty,Unit,5500,4600,5500,1,5980,Cash,2021-01-01 08:08:33


In [8]:
step2_df.to_csv(STEP2_OUT, index=False)
print(f"Step 2 preprocessed dataset saved to: {STEP2_OUT.resolve()}")

Step 2 preprocessed dataset saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\data\processed\step2_preprocessed_dataset.csv


## Step 3: Feature Engineering (Lag + Rolling + Seasonality)

In this step, we transform cleaned transactional data into product-level daily features for forecasting:
- Aggregate transactions to daily product demand.
- Create lag features (1, 7, 14, 28 days).
- Create rolling statistics (7, 14, 28 day windows).
- Add calendar and Sri Lankan seasonality indicators.
- Save a model-ready feature dataset for Step 4 time-based splitting.

In [9]:
STEP3_OUT = PROCESSED_DIR / "step3_feature_engineered_dataset.csv"


def add_sri_lanka_seasonality_flags(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy()

    # Practical season proxies for Sri Lanka retail demand.
    data["is_avurudu_season"] = data["month"].isin([4]).astype(int)
    data["is_vesak_season"] = data["month"].isin([5]).astype(int)
    data["is_year_end_festive"] = data["month"].isin([12]).astype(int)
    data["is_tourist_peak"] = data["month"].isin([12, 1, 2, 3]).astype(int)

    return data


def engineer_step3_features(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    data = frame.copy()

    if "timestamp" not in data.columns:
        raise ValueError("Step 2 data must include a 'timestamp' column.")

    data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
    data = data.dropna(subset=["timestamp", "ProductID", "Quantity"]).copy()

    data["date"] = data["timestamp"].dt.floor("D")

    daily = (
        data.groupby(["ProductID", "ProductName", "date"], as_index=False)
        .agg(
            daily_qty=("Quantity", "sum"),
            daily_sales_lkr=("Total_LKR", "sum"),
            avg_unit_price=("UnitPrice", "mean"),
            txn_count=("TransactionID", "count"),
        )
        .sort_values(["ProductID", "date"])
        .reset_index(drop=True)
    )

    daily["day_of_week"] = daily["date"].dt.dayofweek
    daily["week_of_year"] = daily["date"].dt.isocalendar().week.astype(int)
    daily["month"] = daily["date"].dt.month
    daily["quarter"] = daily["date"].dt.quarter
    daily["is_weekend"] = daily["day_of_week"].isin([5, 6]).astype(int)

    group = daily.groupby("ProductID", group_keys=False)

    for lag in [1, 7, 14, 28]:
        daily[f"lag_qty_{lag}d"] = group["daily_qty"].shift(lag)

    for window in [7, 14, 28]:
        shifted = group["daily_qty"].shift(1)
        daily[f"roll_mean_qty_{window}d"] = (
            shifted.groupby(daily["ProductID"]).rolling(window, min_periods=3).mean().reset_index(level=0, drop=True)
        )
        daily[f"roll_std_qty_{window}d"] = (
            shifted.groupby(daily["ProductID"]).rolling(window, min_periods=3).std().reset_index(level=0, drop=True)
        )

    daily = add_sri_lanka_seasonality_flags(daily)

    rows_before_dropna = len(daily)
    feature_cols = [
        "lag_qty_1d", "lag_qty_7d", "lag_qty_14d", "lag_qty_28d",
        "roll_mean_qty_7d", "roll_mean_qty_14d", "roll_mean_qty_28d",
    ]
    feature_df = daily.dropna(subset=feature_cols).reset_index(drop=True)

    report = {
        "product_day_rows": int(rows_before_dropna),
        "model_ready_rows": int(len(feature_df)),
        "rows_dropped_for_lags": int(rows_before_dropna - len(feature_df)),
        "unique_products": int(feature_df["ProductID"].nunique() if len(feature_df) else 0),
        "date_min": str(feature_df["date"].min()) if len(feature_df) else "NA",
        "date_max": str(feature_df["date"].max()) if len(feature_df) else "NA",
    }

    return feature_df, report

In [10]:
step3_df, step3_report = engineer_step3_features(step2_df)

print("Step 3 Report")
for key, value in step3_report.items():
    print(f"- {key}: {value:,}" if isinstance(value, int) else f"- {key}: {value}")

step3_df.head()

Step 3 Report
- product_day_rows: 760,806
- model_ready_rows: 653,482
- rows_dropped_for_lags: 107,324
- unique_products: 3,833
- date_min: 2021-03-23 00:00:00
- date_max: 2025-12-31 00:00:00


,ProductID,ProductName,date,daily_qty,daily_sales_lkr,avg_unit_price,txn_count,day_of_week,week_of_year,month,quarter,is_weekend,lag_qty_1d,lag_qty_7d,lag_qty_14d,lag_qty_28d,roll_mean_qty_7d,roll_std_qty_7d,roll_mean_qty_14d,roll_std_qty_14d,roll_mean_qty_28d,roll_std_qty_28d,is_avurudu_season,is_vesak_season,is_year_end_festive,is_tourist_peak
0,PI00001,Captain Oats Instant - 500g,2021-08-25,3,12235,1020.0,1,2,34,8,3,0,1.0,1.0,1.0,2.0,2.285714,1.603567,2.142857,1.292412,2.428571,1.596955,0,0,0,0
1,PI00001,Captain Oats Instant - 500g,2021-09-13,4,8870,1070.0,2,0,37,9,3,0,3.0,5.0,3.0,4.0,2.571429,1.511858,2.285714,1.266647,2.464286,1.598197,0,0,0,0
2,PI00001,Captain Oats Instant - 500g,2021-09-29,3,8370,1080.0,1,2,39,9,3,0,4.0,2.0,2.0,2.0,2.428571,1.272418,2.357143,1.336306,2.464286,1.598197,0,0,0,0
3,PI00001,Captain Oats Instant - 500g,2021-10-03,4,12235,1120.0,1,6,39,10,4,1,3.0,2.0,3.0,1.0,2.571429,1.272418,2.428571,1.342460,2.500000,1.598611,0,0,0,0
4,PI00001,Captain Oats Instant - 500g,2021-10-05,1,4910,1140.0,1,1,40,10,4,0,4.0,4.0,1.0,4.0,2.857143,1.345185,2.500000,1.400549,2.607143,1.594883,0,0,0,0


In [11]:
step3_df.to_csv(STEP3_OUT, index=False)
print(f"Step 3 feature dataset saved to: {STEP3_OUT.resolve()}")

Step 3 feature dataset saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\data\processed\step3_feature_engineered_dataset.csv


Step 3 complete. Next: Step 4 time-based train/validation/test split without shuffling.